<p align="center">
  <img src="../assets/prodinno_logo.png" alt="Prodinno" width="200">
</p>

<h4 align="center">Module 0 · Foundations</h4>
<h1 align="center">NumPy & pandas Essentials</h1>
<p align="center"><i>A fast brush-up on arrays and DataFrames before we start modeling</i></p>

---

## Why This Module Exists

Every notebook in this workshop leans on a handful of core libraries: **NumPy** for numeric
arrays and **pandas** for tabular data, plus **Matplotlib**, **Seaborn**, and **Plotly** for
visualization. If you've used them before but it's been a while, this module is a fast
refresher covering just the pieces that show up again and again in the topic notebooks that
follow. It is **not** a deep dive - each topic folder's own `01_eda.ipynb` will go much
further on the specifics that matter for that dataset.

This notebook covers **NumPy** and **pandas**. The companion notebooks in this folder --
`01_matplotlib.ipynb`, `02_seaborn.ipynb`, and `03_plotly.ipynb` - cover the three
visualization libraries. Each of those rebuilds the same synthetic dataset introduced below
from scratch, so every notebook in this folder can be run on its own, in any order.

We'll build one small synthetic dataset here and reuse it for the rest of this notebook.

## 1. NumPy Essentials

NumPy's core object is the **array** (`ndarray`) - a grid of values, all the same dtype,
that supports fast, vectorized math. Every other library in this stack (pandas, scikit-learn,
even Plotly under the hood) is built on top of NumPy arrays.

### 1.1 Creating arrays

In [1]:
import numpy as np

a = np.array([1, 2, 3, 4, 5])                 # from a Python list
b = np.arange(0, 10, 2)                       # like range(), but returns an array
c = np.linspace(0, 1, 5)                      # 5 evenly spaced points between 0 and 1
d = np.zeros((2, 3))                          # a 2x3 array of zeros
e = np.ones((3,))                             # a length-3 array of ones
rng = np.random.default_rng(seed=42)          # a seeded random generator - reproducible
f = rng.normal(loc=0, scale=1, size=(2, 4))   # 2x4 array of standard-normal draws

print("a:", a)
print("b (arange):", b)
print("c (linspace):", c)
print("d (zeros):\n", d)
print("e (ones):", e)
print("f (random normal):\n", f)

a: [1 2 3 4 5]
b (arange): [0 2 4 6 8]
c (linspace): [0.   0.25 0.5  0.75 1.  ]
d (zeros):
 [[0. 0. 0.]
 [0. 0. 0.]]
e (ones): [1. 1. 1.]
f (random normal):
 [[ 0.30471708 -1.03998411  0.7504512   0.94056472]
 [-1.95103519 -1.30217951  0.1278404  -0.31624259]]


### 1.2 Shape, dtype, and reshaping

Every array has a **shape** (its dimensions) and a **dtype** (the type of every element,
since unlike a Python list, all elements share one type). `reshape` lets you rearrange the
same underlying data into a different shape, as long as the total element count matches.

In [2]:
arr = np.arange(12)
print("shape:", arr.shape, "dtype:", arr.dtype, "ndim:", arr.ndim)

grid = arr.reshape(3, 4)     # 12 elements -> a 3x4 grid
print("reshaped to 3x4:\n", grid)

# -1 means "figure this dimension out for me"
print("reshaped to (2, -1):\n", arr.reshape(2, -1))

shape: (12,) dtype: int64 ndim: 1
reshaped to 3x4:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
reshaped to (2, -1):
 [[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]]


### 1.3 Indexing, slicing, and boolean masks

Arrays support the same `[start:stop:step]` slicing as Python lists, plus two extras that
pandas and scikit-learn use constantly: **boolean masks** (select elements where a condition
is true) and **fancy indexing** (select elements by a list of positions).

In [3]:
grid = np.arange(12).reshape(3, 4)
print("grid:\n", grid)
print("row 1:", grid[1])
print("column 2:", grid[:, 2])
print("bottom-right 2x2 block:\n", grid[1:, 2:])

values = np.array([3, -1, 7, 0, -5, 9, 2])
mask = values > 0                       # a boolean array, same shape as `values`
print("mask:", mask)
print("values where mask is True:", values[mask])   # boolean indexing
print("values at positions [0, 2, 5]:", values[[0, 2, 5]])   # fancy indexing

values[values < 0] = 0                  # boolean masks work for assignment too
print("negatives clipped to 0:", values)

grid:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
row 1: [4 5 6 7]
column 2: [ 2  6 10]
bottom-right 2x2 block:
 [[ 6  7]
 [10 11]]
mask: [ True False  True False False  True  True]
values where mask is True: [3 7 9 2]
values at positions [0, 2, 5]: [3 7 9]
negatives clipped to 0: [3 0 7 0 0 9 2]


### 1.4 Vectorization: why loop when you can broadcast?

A Python `for` loop over an array re-evaluates Python bytecode for every element. NumPy's
vectorized operations instead push the loop down into compiled C code, operating on the whole
array at once. On anything beyond toy sizes, the difference is dramatic - and vectorized code
is usually shorter and easier to read too.

In [4]:
import time

n = 2_000_000
x = np.random.default_rng(0).normal(size=n)

start = time.perf_counter()
loop_result = [xi ** 2 + 1 for xi in x]
loop_time = time.perf_counter() - start

start = time.perf_counter()
vec_result = x ** 2 + 1
vec_time = time.perf_counter() - start

print(f"Python loop:  {loop_time:.4f}s")
print(f"Vectorized:   {vec_time:.4f}s")
print(f"Speedup:      {loop_time / vec_time:,.0f}x")

Python loop:  0.9332s
Vectorized:   0.0212s
Speedup:      44x


### 1.5 Broadcasting

**Broadcasting** is the rule set NumPy uses to apply an operation between arrays of
*different* (but compatible) shapes, without you having to manually copy data to match
shapes. Two dimensions are compatible when they're equal, or when one of them is 1. A shape-1
dimension is stretched to match the other array.

$$\underbrace{(3, 4)}_{\text{grid}} \quad \text{and} \quad \underbrace{(4,)}_{\text{row vector}} \;\longrightarrow\; \text{the row vector is treated as shape } (1, 4) \text{, then stretched down to } (3, 4)$$

In [5]:
grid = np.ones((3, 4))          # shape (3, 4)
row_offsets = np.array([10, 20, 30, 40])   # shape (4,)

# NumPy "broadcasts" row_offsets across all 3 rows without us writing a loop
result = grid + row_offsets
print("grid + row_offsets:\n", result)

col_offsets = np.array([[100], [200], [300]])   # shape (3, 1)
print("\ngrid + col_offsets (broadcast the other way):\n", grid + col_offsets)

grid + row_offsets:
 [[11. 21. 31. 41.]
 [11. 21. 31. 41.]
 [11. 21. 31. 41.]]

grid + col_offsets (broadcast the other way):
 [[101. 101. 101. 101.]
 [201. 201. 201. 201.]
 [301. 301. 301. 301.]]


### 1.6 Aggregations and the `axis` parameter

Functions like `.sum()`, `.mean()`, `.std()`, `.min()`, `.max()` collapse an array down to a
single number by default. Pass `axis=` to collapse along just one dimension instead --
`axis=0` collapses *down the rows* (one result per column), `axis=1` collapses *across the
columns* (one result per row). This exact `axis` convention reappears constantly in pandas.

In [6]:
grid = np.arange(12).reshape(3, 4).astype(float)
print("grid:\n", grid)
print("overall mean:", grid.mean())
print("mean per column (axis=0):", grid.mean(axis=0))
print("mean per row    (axis=1):", grid.mean(axis=1))
print("overall std:", round(grid.std(), 3))

grid:
 [[ 0.  1.  2.  3.]
 [ 4.  5.  6.  7.]
 [ 8.  9. 10. 11.]]
overall mean: 5.5
mean per column (axis=0): [4. 5. 6. 7.]
mean per row    (axis=1): [1.5 5.5 9.5]
overall std: 3.452


## 2. Pandas Essentials

Pandas' `DataFrame` is a labeled, 2D table built on top of NumPy arrays - one column per
variable, one row per observation, with an index and column names attached. It's the format
every dataset in this workshop arrives in.

We'll build one small synthetic "store sales" dataset here and reuse it for the rest of this
notebook.

In [7]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 300

regions = rng.choice(["North", "South", "East", "West"], size=n, p=[0.3, 0.3, 0.2, 0.2])
categories = rng.choice(["Electronics", "Grocery", "Apparel"], size=n)
units_sold = rng.poisson(lam=20, size=n)
unit_price = np.round(rng.uniform(5, 200, size=n), 2)
customer_rating = np.clip(rng.normal(loc=4.0, scale=0.7, size=n), 1, 5).round(1)

sales = pd.DataFrame({
    "region": regions,
    "category": categories,
    "units_sold": units_sold,
    "unit_price": unit_price,
    "revenue": np.round(units_sold * unit_price, 2),
    "customer_rating": customer_rating,
})

# Inject a few realistic missing values, so plots involving customer_rating have gaps to handle
missing_idx = rng.choice(sales.index, size=15, replace=False)
sales.loc[missing_idx, "customer_rating"] = np.nan

print(sales.shape)
sales.head()

(300, 6)


,region,category,units_sold,unit_price,revenue,customer_rating
0,East,Grocery,23,74.69,1717.87,3.9
1,South,Apparel,13,56.64,736.32,4.3
2,West,Apparel,24,124.04,2976.96,2.5
3,East,Electronics,21,45.05,946.05,4.0
4,North,Apparel,29,178.23,5168.67,5.0


### 2.1 First look: `.head()`, `.info()`, `.describe()`

In [8]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   region           300 non-null    str    
 1   category         300 non-null    str    
 2   units_sold       300 non-null    int64  
 3   unit_price       300 non-null    float64
 4   revenue          300 non-null    float64
 5   customer_rating  285 non-null    float64
dtypes: float64(3), int64(1), str(2)
memory usage: 18.0 KB


In [9]:
sales.describe()

,units_sold,unit_price,revenue,customer_rating
count,300.000000,300.000000,300.000000,285.000000
mean,19.780000,103.831933,2035.676333,3.871930
std,4.463707,56.902403,1226.919229,0.671176
min,9.000000,5.900000,125.710000,2.100000
25%,17.000000,54.387500,988.990000,3.400000
50%,20.000000,101.535000,1897.355000,3.900000
75%,23.000000,155.502500,2893.740000,4.400000
max,31.000000,199.350000,5866.500000,5.000000


### 2.2 Selecting columns and filtering rows

Square brackets select columns (`df["col"]` for one, `df[["a", "b"]]` for several).
A boolean condition inside square brackets filters rows - exactly the same masking idea
from NumPy Section 1.3, now applied to a labeled table.

In [10]:
prices_only = sales["unit_price"]
subset_cols = sales[["region", "revenue"]]

high_value_sales = sales[sales["revenue"] > 2000]
print(f"{len(high_value_sales)} of {len(sales)} orders had revenue > 2000")

west_electronics = sales[(sales["region"] == "West") & (sales["category"] == "Electronics")]
print(f"{len(west_electronics)} West-region Electronics orders")
west_electronics.head(3)

145 of 300 orders had revenue > 2000
18 West-region Electronics orders


,region,category,units_sold,unit_price,revenue,customer_rating
18,West,Electronics,24,68.17,1636.08,3.8
22,West,Electronics,20,38.22,764.40,4.8
31,West,Electronics,24,67.61,1622.64,3.3


### 2.3 `.loc` vs. `.iloc`

`.loc` selects by **label** (row/column names); `.iloc` selects by **integer position** --
the same distinction as dict-style lookup vs. list-style indexing.

In [11]:
print("loc, by label:\n", sales.loc[0:2, ["region", "revenue"]])
print("\niloc, by position (first 3 rows, first 2 columns):\n", sales.iloc[0:3, 0:2])

loc, by label:
   region  revenue
0   East  1717.87
1  South   736.32
2   West  2976.96

iloc, by position (first 3 rows, first 2 columns):
   region category
0   East  Grocery
1  South  Apparel
2   West  Apparel


### 2.4 Creating and transforming columns

In [12]:
sales["revenue_per_unit"] = sales["revenue"] / sales["units_sold"]
sales["price_tier"] = pd.cut(
    sales["unit_price"], bins=[0, 50, 120, 300], labels=["low", "mid", "high"]
)
sales[["unit_price", "price_tier", "revenue_per_unit"]].head()

,unit_price,price_tier,revenue_per_unit
0,74.69,mid,74.69
1,56.64,mid,56.64
2,124.04,high,124.04
3,45.05,low,45.05
4,178.23,high,178.23


### 2.5 Missing values: `.isna()`, `.fillna()`, `.dropna()`

Real data always has gaps. Pandas represents a missing value as `NaN` (Not a Number), and
gives you three basic tools: **detect** it, **fill** it with a reasonable value, or **drop**
the rows/columns that contain it.

In [13]:
print("missing values per column:\n", sales.isna().sum())

rating_filled = sales["customer_rating"].fillna(sales["customer_rating"].mean())
print(f"\nmean rating before fill: {sales['customer_rating'].mean():.3f}")
print(f"missing count before fill: {sales['customer_rating'].isna().sum()}, after fill: {rating_filled.isna().sum()}")

sales_complete = sales.dropna(subset=["customer_rating"])
print(f"\nrows before dropna: {len(sales)}, after dropna: {len(sales_complete)}")

missing values per column:
 region               0
category             0
units_sold           0
unit_price           0
revenue              0
customer_rating     15
revenue_per_unit     0
price_tier           0
dtype: int64

mean rating before fill: 3.872
missing count before fill: 15, after fill: 0

rows before dropna: 300, after dropna: 285


### 2.6 `.groupby()` - split, apply, combine

`.groupby()` splits the DataFrame into groups by one or more columns, applies an aggregation
to each group independently, then combines the results back into one table. This is the same
pattern SQL's `GROUP BY` and Excel's pivot tables use.

In [14]:
region_summary = sales.groupby("region").agg(
    total_revenue=("revenue", "sum"),
    avg_rating=("customer_rating", "mean"),
    n_orders=("revenue", "count"),
).sort_values("total_revenue", ascending=False)

region_summary

,total_revenue,avg_rating,n_orders
region,,,
North,206362.85,3.944086,97
South,186424.22,3.859524,88
West,115441.30,3.774510,57
East,102474.53,3.859649,58


### 2.7 `.merge()` - joining two tables

Real projects rarely live in one table. `.merge()` joins two DataFrames on a shared key
column, just like a SQL `JOIN`. Here we attach a small regional-manager lookup table onto
every sales row.

In [15]:
region_managers = pd.DataFrame({
    "region": ["North", "South", "East", "West"],
    "manager": ["Aditi", "Rohan", "Meera", "Farhan"],
})

sales_with_manager = sales.merge(region_managers, on="region", how="left")
sales_with_manager[["region", "manager", "revenue"]].head()

,region,manager,revenue
0,East,Meera,1717.87
1,South,Rohan,736.32
2,West,Farhan,2976.96
3,East,Meera,946.05
4,North,Aditi,5168.67


## 3. Which Tool, When?

The original Module 0 decision guide is split across this folder's four notebooks, one entry
per notebook. Here are the entries for the two libraries covered in this one:

- **NumPy** - the numeric engine underneath everything else; reach for it directly whenever
  you're doing math on arrays rather than working with labeled tabular data.
- **pandas** - the default for loading, cleaning, filtering, grouping, and joining tabular
  data; almost every notebook in this workshop starts here.

See `01_matplotlib.ipynb`, `02_seaborn.ipynb`, and `03_plotly.ipynb` for the visualization
entries.

## Summary

- **NumPy**: array creation, shape/dtype/reshape, indexing/slicing/boolean masks,
  vectorization vs. Python loops, broadcasting, and `axis`-aware aggregations.
- **pandas**: building and inspecting a DataFrame, column selection and filtering,
  `.loc`/`.iloc`, creating/transforming columns, handling missing values, `.groupby()`, and
  `.merge()`.
- When to reach for NumPy directly vs. pandas for labeled tabular work.

Continue to `01_matplotlib.ipynb`, `02_seaborn.ipynb`, and `03_plotly.ipynb` for the
visualization half of this refresher - each one rebuilds this same synthetic "store sales"
dataset from scratch, so they can be run in any order.